In [9]:
# %pip install kagglehub
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, PolynomialFeatures, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report,confusion_matrix, ConfusionMatrixDisplay, f1_score, roc_curve,roc_auc_score
from sklearn.linear_model import ElasticNet,Lasso,Ridge
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_regression, SelectKBest
from scipy import stats
from sklearn.feature_selection import f_regression
from sklearn.datasets import load_breast_cancer
from collections import Counter
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn.dummy import DummyClassifier
from sklearn.dummy import DummyClassifier
from sklearn.compose import ColumnTransformer
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.model_selection import RandomizedSearchCV
import graphviz
import math
warnings.filterwarnings('ignore')
df = pd.read_csv('../../Datasets/fromclass/vehicle_emission_dataset.csv')

In [10]:
X = df.drop(['CO2 Emissions'],axis=1)
y = df['CO2 Emissions']


categorical_cols = X.select_dtypes(include=["object"]).columns
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns


In [ ]:



numerical_pipeLine = Pipeline([
  ('imputer', SimpleImputer(strategy='most_frequent')),
  ('scaler', StandardScaler())
])
calegorical_pipeLine = Pipeline([
  ('imputer', SimpleImputer(strategy='most_frequent')),
  ('encoder', OneHotEncoder(handle_unknown='ignore'))
])


preprocessor = ColumnTransformer([
  ('num',numerical_pipeLine, numerical_cols),
  ('cat',calegorical_pipeLine, categorical_cols)
])


pipe = Pipeline([
  ('preprocessor',preprocessor),
  ('model', RandomForestRegressor())
])



random_search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions={
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 10, 20],
        "model__min_samples_split": [2, 5],
        "model__min_samples_leaf": [1, 2, 4],
    },
    n_iter=10,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

random_search.fit(X_train, y_train)

print("Best params:", random_search.best_params_)

best_model = random_search.best_estimator_

y_pred = best_model.predict(X_test)

print("R2:", r2_score(y_test, y_pred))

Best params: {'model__n_estimators': 100, 'model__min_samples_split': 2, 'model__min_samples_leaf': 2, 'model__max_depth': 10}
R2: 0.8485850803798956


In [12]:
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# pipe.fit(X_train, y_train)
# prediction = pipe.predict(X_test)

# encoded_cols = pipe.named_steps['preprocessor'].named_transformers_['cat']['encoder'].get_feature_names_out(categorical_cols)
# # print(encoded_cols)
# mse = mean_squared_error(y_test, prediction)
# rmse = np.sqrt(mse)
# r2 = r2_score(y_test,prediction)

# print(f'mse score {mse}')
# print()
# print(f'rmse score {rmse}')
# print()
# print(f'R2 Score {r2}')


# # joblib.dump(pipeline,"Vehicle_emission")


In [13]:
# from sklearn.preprocessing import PowerTransformer

# # Add this to your numerical_pipeline
# numerical_pipeline = Pipeline([
#     ('imputer', SimpleImputer(strategy='most_frequent')),
#     ('skew_fix', PowerTransformer(method='yeo-johnson')), # Handles skewness
#     ('scaler', StandardScaler())
# ])